# Introduction

Training a model

In [1]:
# (optionnel) si PyTorch n'est pas présent, décommente :
# !pip install torch==2.3.1 --quiet

import os
import math
from collections import Counter
from typing import List, Tuple, Dict

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE


device(type='cpu')

## Chargement des données brutes

In [2]:
# 3) Charger les données
EN_PATH = "../data/raw/small_vocab_en.txt"
FR_PATH = "../data/raw/small_vocab_fr.txt"

# Si tu veux réutiliser ta fonction utilitaire existante:
# (elle lit un chemin et renvoie les lignes du fichier)
from helper import load_data  # :contentReference[oaicite:2]{index=2}

raw_en = [s.strip() for s in load_data(EN_PATH) if s.strip()]
raw_fr = [s.strip() for s in load_data(FR_PATH) if s.strip()]

assert len(raw_en) == len(raw_fr), "Les fichiers EN/FR doivent avoir le même nombre de lignes."
len(raw_en), raw_en[:3], raw_fr[:3]


(137860,
 ['new jersey is sometimes quiet during autumn , and it is snowy in april .',
  'the united states is usually chilly during july , and it is usually freezing in november .',
  'california is usually quiet during march , and it is usually hot in june .'],
 ["new jersey est parfois calme pendant l' automne , et il est neigeux en avril .",
  'les états-unis est généralement froid en juillet , et il gèle habituellement en novembre .',
  'california est généralement calme en mars , et il est généralement chaud en juin .'])

## Pré-traitement minimal & jeux d’entrainement/validation

In [3]:
import random
random.seed(42)

def basic_norm(s: str) -> str:
    # Choix minimal : lower + espaces autour de la ponctuation simple
    s = s.lower().strip()
    for p in [".", ",", "!", "?", ":", ";"]:
        s = s.replace(p, f" {p} ")
    s = " ".join(s.split())
    return s

en = [basic_norm(s) for s in raw_en]
fr = [basic_norm(s) for s in raw_fr]

# Ajouter marqueurs BOS/EOS côté cible (FR)
fr_in  = [("<bos> " + s).strip() for s in fr]
fr_out = [(s + " <eos>").strip() for s in fr]

# Split train/val
idx = list(range(len(en)))
random.shuffle(idx)
split = int(0.9 * len(idx))
train_idx, val_idx = idx[:split], idx[split:]

train_en = [en[i] for i in train_idx]
train_fr_in = [fr_in[i] for i in train_idx]
train_fr_out = [fr_out[i] for i in train_idx]

val_en = [en[i] for i in val_idx]
val_fr_in = [fr_in[i] for i in val_idx]
val_fr_out = [fr_out[i] for i in val_idx]

len(train_en), len(val_en)


(124074, 13786)

## Vocabulaire & tokenisation

In [4]:
PAD, UNK, BOS, EOS = "<pad>", "<unk>", "<bos>", "<eos>"

def build_vocab(sentences: List[str], min_freq: int = 1, specials=(PAD, UNK, BOS, EOS)) -> Tuple[Dict[str,int], Dict[int,str]]:
    cnt = Counter()
    for s in sentences:
        cnt.update(s.split())
    # specials au début
    itos = list(specials)
    for tok, c in cnt.items():
        if c >= min_freq and tok not in specials:
            itos.append(tok)
    stoi = {w:i for i,w in enumerate(itos)}
    return stoi, {i:w for w,i in stoi.items()}

# Vocab source = anglais
src_stoi, src_itos = build_vocab(train_en, min_freq=1, specials=(PAD, UNK, BOS, EOS))
# Vocab cible = français (sur fr_in + fr_out pour couvrir BOS/EOS et tous tokens)
tgt_stoi, tgt_itos = build_vocab(train_fr_in + train_fr_out, min_freq=1, specials=(PAD, UNK, BOS, EOS))

PAD_IDX = src_stoi[PAD]  # on suppose même index pour pad source/target
assert PAD_IDX == tgt_stoi[PAD]

len(src_stoi), len(tgt_stoi)


(206, 356)

## Numérisation & padding

In [5]:
def numericalize(s: str, stoi: Dict[str,int]) -> List[int]:
    return [stoi.get(tok, stoi[UNK]) for tok in s.split()]

def pad_to_len(ids: List[int], max_len: int, pad_id: int) -> List[int]:
    if len(ids) >= max_len:
        return ids[:max_len]
    return ids + [pad_id] * (max_len - len(ids))

def compute_max_lens(src: List[str], tgt_in: List[str], tgt_out: List[str]) -> Tuple[int,int]:
    src_max = max(len(s.split()) for s in src)
    tgt_max = max(len(s.split()) for s in tgt_out)  # la sortie doit pouvoir contenir <eos>
    return src_max, tgt_max

SRC_MAX_LEN, TGT_MAX_LEN = compute_max_lens(train_en, train_fr_in, train_fr_out)
SRC_MAX_LEN, TGT_MAX_LEN


(17, 24)

## Dataset & DataLoader

In [6]:
class NMTDataset(Dataset):
    def __init__(self, src_sents, tgt_in_sents, tgt_out_sents, src_stoi, tgt_stoi,
                 src_max_len, tgt_max_len, pad_idx):
        self.src = src_sents
        self.tgt_in = tgt_in_sents
        self.tgt_out = tgt_out_sents
        self.src_stoi = src_stoi
        self.tgt_stoi = tgt_stoi
        self.src_max = src_max_len
        self.tgt_max = tgt_max_len
        self.pad = pad_idx

    def __len__(self):
        return len(self.src)

    def __getitem__(self, idx):
        x = pad_to_len(numericalize(self.src[idx], self.src_stoi), self.src_max, self.pad)
        y_in  = pad_to_len(numericalize(self.tgt_in[idx],  self.tgt_stoi), self.tgt_max, self.pad)
        y_out = pad_to_len(numericalize(self.tgt_out[idx], self.tgt_stoi), self.tgt_max, self.pad)
        return torch.tensor(x, dtype=torch.long), torch.tensor(y_in, dtype=torch.long), torch.tensor(y_out, dtype=torch.long)

train_ds = NMTDataset(train_en, train_fr_in, train_fr_out, src_stoi, tgt_stoi, SRC_MAX_LEN, TGT_MAX_LEN, PAD_IDX)
val_ds   = NMTDataset(val_en,   val_fr_in,   val_fr_out,   src_stoi, tgt_stoi, SRC_MAX_LEN, TGT_MAX_LEN, PAD_IDX)

BATCH_SIZE = 128
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE)
len(train_ds), len(val_ds)


(124074, 13786)

## Simple model” (GRU + projection par pas) — baseline

In [7]:
class SimpleGRUModel(nn.Module):
    def __init__(self, src_vocab, tgt_vocab, emb_dim=128, hid_dim=256, pad_idx=0):
        super().__init__()
        self.src_embed = nn.Embedding(src_vocab, emb_dim, padding_idx=pad_idx)
        self.gru = nn.GRU(emb_dim, hid_dim, batch_first=True)
        self.fc  = nn.Linear(hid_dim, tgt_vocab)

    def forward(self, src_tokens):  # (B, S)
        x = self.src_embed(src_tokens)     # (B, S, E)
        h, _ = self.gru(x)                 # (B, S, H)
        logits = self.fc(h)               # (B, S, Vtgt)
        return logits

simple_model = SimpleGRUModel(len(src_stoi), len(tgt_stoi), emb_dim=128, hid_dim=256, pad_idx=PAD_IDX).to(DEVICE)
sum(p.numel() for p in simple_model.parameters())/1e6


0.414308

## Boucle d’entraînement générique (CE + mask du PAD)

In [8]:
import torch.nn.functional as F

def sequence_ce_loss(logits, targets, pad_idx):
    # logits: (B, T, V), targets: (B, T)
    # Certains slices rendent non contigu → reshape plutôt que view
    logits = logits.reshape(-1, logits.size(-1))
    targets = targets.reshape(-1)
    return F.cross_entropy(logits, targets, ignore_index=pad_idx)


def run_epoch(model, loader, optimizer=None):
    train_mode = optimizer is not None
    model.train(train_mode)
    total, n = 0.0, 0
    for x, y_in, y_out in loader:
        x, y_in, y_out = x.to(DEVICE), y_in.to(DEVICE), y_out.to(DEVICE)
        # Pour le simple_model, on ignore y_in : on projette directement chaque pas source
        logits = model(x)  # (B, S, Vtgt) — on entraînera à prédire y_out aligné
        # on tronque aux mêmes longueurs si besoin
        T = min(logits.size(1), y_out.size(1))
        loss = sequence_ce_loss(logits[:, :T, :], y_out[:, :T], PAD_IDX)

        if train_mode:
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        total += loss.item() * x.size(0)
        n += x.size(0)
    return total / n

def evaluate(model, loader):
    with torch.no_grad():
        return run_epoch(model, loader, optimizer=None)


## Entraînement rapide du “simple model”

In [9]:
EPOCHS = 5
optimizer = optim.Adam(simple_model.parameters(), lr=0.01)

for epoch in range(1, EPOCHS+1):
    tr_loss = run_epoch(simple_model, train_dl, optimizer)
    va_loss = evaluate(simple_model, val_dl)
    print(f"[Simple] Epoch {epoch:02d} | train {tr_loss:.3f} | val {va_loss:.3f}")


[Simple] Epoch 01 | train 0.702 | val 0.613
[Simple] Epoch 02 | train 0.636 | val 0.657
[Simple] Epoch 03 | train 0.679 | val 0.724
[Simple] Epoch 04 | train 0.764 | val 0.747
[Simple] Epoch 05 | train 0.755 | val 0.759


## Modèle “Embedding” (identique mais configurable)

In [10]:
class EmbeddingGRUModel(SimpleGRUModel):
    def __init__(self, src_vocab, tgt_vocab, emb_dim=256, hid_dim=256, pad_idx=0):
        super().__init__(src_vocab, tgt_vocab, emb_dim, hid_dim, pad_idx)

embed_model = EmbeddingGRUModel(len(src_stoi), len(tgt_stoi), emb_dim=256, hid_dim=256, pad_idx=PAD_IDX).to(DEVICE)
optimizer = optim.Adam(embed_model.parameters(), lr=3e-3)

for epoch in range(1, 4):
    tr_loss = run_epoch(embed_model, train_dl, optimizer)
    va_loss = evaluate(embed_model, val_dl)
    print(f"[Embed] Epoch {epoch:02d} | train {tr_loss:.3f} | val {va_loss:.3f}")


[Embed] Epoch 01 | train 0.677 | val 0.466
[Embed] Epoch 02 | train 0.434 | val 0.416
[Embed] Epoch 03 | train 0.403 | val 0.405


## Encodeur–Décodeur (seq2seq) avec attention

In [11]:
class Encoder(nn.Module):
    def __init__(self, src_vocab, emb_dim=256, hid_dim=256, pad_idx=0, bidirectional=False):
        super().__init__()
        self.embed = nn.Embedding(src_vocab, emb_dim, padding_idx=pad_idx)
        self.bidirectional = bidirectional
        self.gru = nn.GRU(emb_dim, hid_dim, batch_first=True, bidirectional=bidirectional)
        self.hid_dim = hid_dim * (2 if bidirectional else 1)

    def forward(self, src):  # (B, S)
        emb = self.embed(src)           # (B, S, E)
        outputs, h = self.gru(emb)      # outputs: (B, S, H*dir), h: (dir, B, H)
        if self.bidirectional:
            # concat des deux directions pour état final (B, H*2)
            h = torch.cat([h[-2], h[-1]], dim=1).unsqueeze(0)  # (1,B,2H)
        return outputs, h  # outputs pour attention

class BahdanauAttention(nn.Module):
    def __init__(self, enc_hid, dec_hid):
        super().__init__()
        self.W1 = nn.Linear(enc_hid, dec_hid)
        self.W2 = nn.Linear(dec_hid, dec_hid)
        self.v  = nn.Linear(dec_hid, 1)

    def forward(self, enc_outputs, hidden, mask):  # enc_outputs: (B,S,Henc), hidden: (1,B,Hdec)
        # score = v^T tanh(W1*enc + W2*h_t)
        B, S, Henc = enc_outputs.size()
        hidden = hidden.transpose(0,1)  # (B,1,Hdec)
        score = self.v(torch.tanh(self.W1(enc_outputs) + self.W2(hidden).expand(-1,S,-1))).squeeze(-1)  # (B,S)
        score = score.masked_fill(mask == 0, -1e9)
        attn = torch.softmax(score, dim=-1)  # (B,S)
        ctx = torch.bmm(attn.unsqueeze(1), enc_outputs).squeeze(1)  # (B,Henc)
        return ctx, attn

class Decoder(nn.Module):
    def __init__(self, tgt_vocab, emb_dim=256, enc_hid=256, dec_hid=256, pad_idx=0):
        super().__init__()
        self.embed = nn.Embedding(tgt_vocab, emb_dim, padding_idx=pad_idx)
        self.gru = nn.GRU(emb_dim + enc_hid, dec_hid, batch_first=True)
        self.attn = BahdanauAttention(enc_hid, dec_hid)
        self.fc = nn.Linear(dec_hid, tgt_vocab)
        self.dec_hid = dec_hid

    def forward(self, y_prev, hidden, enc_outputs, src_mask):
        # y_prev: (B,), hidden: (1,B,Hd), enc_outputs: (B,S,Henc)
        emb = self.embed(y_prev).unsqueeze(1)  # (B,1,E)
        ctx, _ = self.attn(enc_outputs, hidden, src_mask)  # (B,Henc)
        x = torch.cat([emb, ctx.unsqueeze(1)], dim=-1)     # (B,1,E+Henc)
        out, hidden = self.gru(x, hidden)                  # out: (B,1,Hd)
        logits = self.fc(out.squeeze(1))                   # (B,Vtgt)
        return logits, hidden

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, pad_idx, bos_idx, eos_idx, max_len):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.pad = pad_idx
        self.bos = bos_idx
        self.eos = eos_idx
        self.max_len = max_len
        # tailles utiles
        self.enc_hid = encoder.hid_dim

    def make_src_mask(self, src):
        # 1 pour tokens réels, 0 pour PAD
        return (src != self.pad).long()

    def forward(self, src, tgt_in, teacher_forcing=True):
        # Encode
        enc_outputs, h = self.encoder(src)
        src_mask = self.make_src_mask(src)

        B, T = tgt_in.size()
        logits = []
        hidden = h  # (1,B,Hdec)  (si encoder bi, adapter dims)
        y = tgt_in[:,0]  # <bos>

        for t in range(1, T):
            logit, hidden = self.decoder(y, hidden, enc_outputs, src_mask)
            logits.append(logit.unsqueeze(1))
            if teacher_forcing:
                y = tgt_in[:,t]
            else:
                y = logit.argmax(dim=-1)

        return torch.cat(logits, dim=1)  # (B, T-1, V)

    @torch.no_grad()
    def translate(self, src, max_len=None):
        self.eval()
        if max_len is None:
            max_len = self.max_len

        enc_outputs, h = self.encoder(src)
        src_mask = self.make_src_mask(src)
        B = src.size(0)
        y = torch.full((B,), self.bos, dtype=torch.long, device=src.device)
        hidden = h
        out_ids = []

        for _ in range(max_len):
            logit, hidden = self.decoder(y, hidden, enc_outputs, src_mask)
            y = logit.argmax(dim=-1)
            out_ids.append(y.unsqueeze(1))
        return torch.cat(out_ids, dim=1)  # (B, T)


## Instanciation Seq2Seq (encodeur unidirectionnel)

In [12]:
EMB_DIM = 256
HID_DIM = 256

encoder = Encoder(len(src_stoi), emb_dim=EMB_DIM, hid_dim=HID_DIM, pad_idx=PAD_IDX, bidirectional=False).to(DEVICE)
decoder = Decoder(len(tgt_stoi), emb_dim=EMB_DIM, enc_hid=encoder.hid_dim, dec_hid=HID_DIM, pad_idx=PAD_IDX).to(DEVICE)

model_encdec = Seq2Seq(
    encoder, decoder,
    pad_idx=PAD_IDX,
    bos_idx=tgt_stoi[BOS],
    eos_idx=tgt_stoi[EOS],
    max_len=TGT_MAX_LEN
).to(DEVICE)

sum(p.numel() for p in model_encdec.parameters())/1e6


1.353317

## Entraînement Seq2Seq (teacher forcing)

In [13]:
def train_seq2seq(model, loader, val_loader, epochs=6, lr=1e-3, tf_ratio=1.0):
    opt = optim.Adam(model.parameters(), lr=lr)
    best_val = math.inf
    for ep in range(1, epochs+1):
        model.train()
        total, n = 0.0, 0
        for src, tgt_in, tgt_out in loader:
            src, tgt_in, tgt_out = src.to(DEVICE), tgt_in.to(DEVICE), tgt_out.to(DEVICE)
            # teacher forcing toujours on ici (simple)
            logits = model(src, tgt_in, teacher_forcing=True)  # (B, T-1, V)
            # aligner avec tgt_out[:,1:] (on ne prédit pas le <bos>)
            loss = sequence_ce_loss(logits, tgt_out[:,1:], PAD_IDX)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            total += loss.item() * src.size(0)
            n += src.size(0)

        val_loss = evaluate_seq2seq(model, val_loader)
        print(f"[EncDec] Epoch {ep:02d} | train {total/n:.3f} | val {val_loss:.3f}")
        if val_loss < best_val:
            best_val = val_loss
            torch.save(model.state_dict(), "best_seq2seq.pt")

def evaluate_seq2seq(model, loader):
    model.eval()
    total, n = 0.0, 0
    with torch.no_grad():
        for src, tgt_in, tgt_out in loader:
            src, tgt_in, tgt_out = src.to(DEVICE), tgt_in.to(DEVICE), tgt_out.to(DEVICE)
            logits = model(src, tgt_in, teacher_forcing=True)
            loss = sequence_ce_loss(logits, tgt_out[:,1:], PAD_IDX)
            total += loss.item() * src.size(0)
            n += src.size(0)
    return total / n

train_seq2seq(model_encdec, train_dl, val_dl, epochs=6, lr=1e-3)


[EncDec] Epoch 01 | train 0.412 | val 0.070
[EncDec] Epoch 02 | train 0.056 | val 0.047
[EncDec] Epoch 03 | train 0.038 | val 0.036
[EncDec] Epoch 04 | train 0.030 | val 0.030
[EncDec] Epoch 05 | train 0.025 | val 0.027
[EncDec] Epoch 06 | train 0.023 | val 0.026


## Variante bidirectionnelle (encoder bi-GRU)

In [14]:
encoder_bd = Encoder(
    len(src_stoi), emb_dim=EMB_DIM, hid_dim=HID_DIM,
    pad_idx=PAD_IDX, bidirectional=True
).to(DEVICE)

decoder_bd = Decoder(
    len(tgt_stoi), emb_dim=EMB_DIM,
    enc_hid=encoder_bd.hid_dim,      # = 2*HID_DIM
    dec_hid=encoder_bd.hid_dim,      # = 2*HID_DIM  <<< CHANGEMENT ICI
    pad_idx=PAD_IDX
).to(DEVICE)

model_bd = Seq2Seq(
    encoder_bd, decoder_bd,
    pad_idx=PAD_IDX, bos_idx=tgt_stoi[BOS], eos_idx=tgt_stoi[EOS],
    max_len=TGT_MAX_LEN
).to(DEVICE)

train_seq2seq(model_bd, train_dl, val_dl, epochs=6, lr=1e-3)


[EncDec] Epoch 01 | train 0.247 | val 0.045
[EncDec] Epoch 02 | train 0.034 | val 0.031
[EncDec] Epoch 03 | train 0.025 | val 0.027
[EncDec] Epoch 04 | train 0.021 | val 0.026
[EncDec] Epoch 05 | train 0.019 | val 0.023
[EncDec] Epoch 06 | train 0.016 | val 0.022


## Final model”

In [16]:
EMB_DIM_FINAL = 300
HID_DIM_FINAL = 384
DROPOUT = 0.2

class EncoderDrop(nn.Module):
    def __init__(self, src_vocab, emb_dim, hid_dim, pad_idx, bidirectional=False, dropout=0.2):
        super().__init__()
        self.embed = nn.Embedding(src_vocab, emb_dim, padding_idx=pad_idx)
        self.drop = nn.Dropout(dropout)
        self.bidirectional = bidirectional
        self.gru = nn.GRU(emb_dim, hid_dim, batch_first=True, bidirectional=bidirectional)
        self.hid_dim = hid_dim * (2 if bidirectional else 1)

    def forward(self, src):
        emb = self.drop(self.embed(src))
        outputs, h = self.gru(emb)
        if self.bidirectional:
            h = torch.cat([h[-2], h[-1]], dim=1).unsqueeze(0)
        return outputs, h

class DecoderDrop(Decoder):
    def __init__(self, tgt_vocab, emb_dim, enc_hid, dec_hid, pad_idx, dropout=0.2):
        super().__init__(tgt_vocab, emb_dim, enc_hid, dec_hid, pad_idx)
        self.drop = nn.Dropout(dropout)
        # override embed with dropout wrapper if desired
        self.embed = nn.Embedding(tgt_vocab, emb_dim, padding_idx=pad_idx)
    def forward(self, y_prev, hidden, enc_outputs, src_mask):
        emb = self.embed(y_prev).unsqueeze(1)
        emb = self.drop(emb)
        return super().forward(y_prev, hidden, enc_outputs, src_mask)

encoder_final = EncoderDrop(len(src_stoi), EMB_DIM_FINAL, HID_DIM_FINAL, PAD_IDX, bidirectional=True, dropout=DROPOUT).to(DEVICE)
decoder_final = Decoder(len(tgt_stoi), EMB_DIM_FINAL, encoder_final.hid_dim, HID_DIM_FINAL, PAD_IDX).to(DEVICE)
model_final = Seq2Seq(encoder_final, decoder_final, pad_idx=PAD_IDX, bos_idx=tgt_stoi[BOS], eos_idx=tgt_stoi[EOS], max_len=TGT_MAX_LEN).to(DEVICE)

train_seq2seq(model_final, train_dl, val_dl, epochs=10, lr=8e-4)


RuntimeError: mat1 and mat2 shapes cannot be multiplied (128x768 and 384x384)

## Décodage (greedy) & post-traitement

In [ ]:
def ids_to_sentence(ids: List[int], itos: Dict[int,str]) -> str:
    toks = []
    for i in ids:
        tok = itos[int(i)]
        if tok == EOS:
            break
        if tok in (PAD, BOS):
            continue
        toks.append(tok)
    # recoller la ponctuation simple proprement
    s = " ".join(toks)
    for p in [" .", " ,", " !", " ?", " ;", " :"]:
        s = s.replace(p, p[1:])
    return s.strip()

@torch.no_grad()
def translate_sentence(model: Seq2Seq, sentence_en: str) -> str:
    model.eval()
    x = pad_to_len(numericalize(basic_norm(sentence_en), src_stoi), SRC_MAX_LEN, PAD_IDX)
    x = torch.tensor(x, dtype=torch.long, device=DEVICE).unsqueeze(0)  # (1,S)
    out = model.translate(x, max_len=TGT_MAX_LEN)  # (1,T)
    return ids_to_sentence(out[0].tolist(), tgt_itos)

# Exemples
for s in ["it is cold in new york .", "how are you ?", "the cat is on the table ."]:
    print(s, "→", translate_sentence(model_final, s))


## Sauvegarde & rechargement (format PyTorch .pt)

In [ ]:
# Sauvegarder
torch.save({
    "model_state": model_final.state_dict(),
    "src_stoi": src_stoi,
    "tgt_stoi": tgt_stoi,
    "src_itos": src_itos,
    "tgt_itos": tgt_itos,
    "src_max_len": SRC_MAX_LEN,
    "tgt_max_len": TGT_MAX_LEN,
}, "nmt_final.pt")

# Recharger (exemple)
ckpt = torch.load("nmt_final.pt", map_location=DEVICE)
encoder_reload = EncoderDrop(len(ckpt["src_stoi"]), EMB_DIM_FINAL, HID_DIM_FINAL, PAD_IDX, bidirectional=True, dropout=DROPOUT).to(DEVICE)
decoder_reload = Decoder(len(ckpt["tgt_stoi"]), EMB_DIM_FINAL, encoder_reload.hid_dim, HID_DIM_FINAL, PAD_IDX).to(DEVICE)
model_reload = Seq2Seq(encoder_reload, decoder_reload, pad_idx=PAD_IDX, bos_idx=ckpt["tgt_stoi"][BOS], eos_idx=ckpt["tgt_stoi"][EOS], max_len=ckpt["tgt_max_len"]).to(DEVICE)
model_reload.load_state_dict(ckpt["model_state"])
